# Computing the Distance Grid

In this tutorial, one of the most useful and foundational calculations in mofography, the distance grid. The distance grid refers to a 3D array of floating point values, where the value in each voxel corresponds to the shortest distance to the nearest framework surface. Of course, the definition of framework surface requires assigning some radii to the atoms of the material. By default, mofography uses the covalent radii of the atoms, but you can also use the van der Waals radii or some other mixed and matched set of radii, as we will see shortly. So, in this tutorial we will see the following:

- How to compute the distance grid
- What are the different options while computing the distance grid
- How to visualize the distance grid
  - Computing a distance isosurface
- How to overlay it with the structure visualization from the earlier tutorial. 
- Where to go next

The first few steps are same as before, so we won't be explaining them in detail here, please refer to the Basic structure IO tutorial for more details.

## Imports and paths

In [ ]:
import sys
import os
# * Put your path here
path_to_mgr = os.path.abspath(
    'c:/Users/Arun Gopalan/OneDrive - The University of Manchester/my_cloned_repos/mofography')
sys.path.append(path_to_mgr)

import mofography as mgr

path_to_cif = os.path.join(path_to_mgr, 'examples',
                           'TestCIFs', 'P1-IRMOF-1.cif')


## Read in a structure

In [ ]:
from ase.io import read 
mat_atoms = read(path_to_cif)
mat_updated = mgr.update_with_covalent_bonds(mat_atoms)

## Computing the distance grid
The most important parameter in computing the grid is the grid spacing. This is the fineness of your grid, and is input as the distance between adjacent voxels in angstroms. The finer the grid, the more accurate the calculations, although, keep in mind that,
- when you reduce the grid spacing by a factor of 2, the number of voxels in the grid increases by a factor of 8.
- all the subsequent calculations would also be more expensive, since the grid is larger.

### Calculation under the hood
- On a CPU:
mofography uses Dask to paralellize the grid calcuation for efficiency. Hence, the returned object is a Dask array, which can be converted to a numpy array using the `compute()` method. On top of that, mofography borrows the AABB cell list algorithm from Freud, to further speed up the calculation manifold. 

Some information about the function

In [ ]:
help(mgr.dgrid_from_atoms)

Let's compute a 0.25 $\AA$ grid for IRMOF-1

In [ ]:
dgrid = mgr.dgrid_from_atoms(mat_atoms, spacing=0.25)

Let's look at this returned grid

In [ ]:
dgrid

as I was mentioning earlier, this is a Dask array, so we can convert it to a numpy array using the `compute()` method

In [ ]:
dgrid.compute()

### Tips and tricks #1
In parallel computing, it is pivotal to divide the calculation into pieces of the right size. If the pieces are too small, the overhead of the parallelization is too high, and if the pieces are too large, the parallelization is not effective. The `chunk_size` parameter in the `compute()` method allows you to specify the size of the chunks. The default value is 50000 which is a good starting point. If you are running into memory issues, you can reduce this value. If your RAM is not being fully utilized, you can increase this value, if your RAM is too small, you may want to reduce this value.

### Tips and tricks #2
By default, mofography uses a point probe, meaning it has no size. If you'd like a different probe, like argon, you can specify it using the `probe_radius` parameter. We will leave that as an exercise.

### Tips and tricks #3
If you have a lot of atoms in your unit cell, you may want to reduce the n_neighbors parameter, it will make the calculation of the AABB cell list faster. The default value is 10, which is a good starting point. Since we want the probe-to-framework, surface-to-surface distance, and not the center-to-center distance, we should consider a few center-to-center neighbors while checking for the one with the shortest surface-to-surface distance.

- On a GPU (+CPU):
If you have a powerful nvidia GPU, we might be able to take advantage of that to speed up the calculation. The GPU routine is quite similar to the CPU routine, but doesn't use the cell list algorithm, for memory management reasons. Essentially, it breaks down the calculation onto different cores of the CPU, and then each core talks to the GPU independently to do the calculation. It uses a CuPy backend to do the calculations on the GPU. Unlike the CPU version, it returns a CuPy array. You can convert it to a numpy array using the `asnumpy()` function. Still, you might get a speed up over the CPU routine. Let's look at it in action


In [ ]:
help(mgr.dgrid_from_atoms_gpu)


In [ ]:
dgrid_gpu = mgr.dgrid_from_atoms_gpu(mat_atoms, spacing=0.25)

In [ ]:
dgrid_gpu.device

In [ ]:
import cupy as cp
cp.asnumpy(dgrid_gpu)

### Tips and tricks #4

All the CPU cores need to be able to talk to the GPU at the same time, if your GPU memory is low, this might result in a segmentation fault. But, not to worry, simply reduce the 'block_size' parameter and you should be good to go. Not that, as before, making the block size too small could result in computational bottlenecks in memory instead of the actual computation, resulting in worse performance.

## Visualizing the distance grid

Usually plotting a 3D volume is computationally expensive and often unclear. Instead, we can plot and isosurface of the distance grid, which is a surface that is at a certain constant distance from the framework surface. This is a much more intuitive way to visualize the distance grid. Let's look at how to do that in mofography.

In [ ]:
help(mgr.plot_surface_dgrid)

let's plot an isosurface at 2.5 $\AA$ from the framework surface

In [ ]:
isosurface_trace = mgr.plot_surface_dgrid(cp.asnumpy(dgrid_gpu), ase_atoms = mat_atoms , isovalue=2.5, color='turquoise', opacity=0.6)
isosurface_trace

Let's make a plolty figure, and add the isosurface and the structure to it. Please refer to the earlier tutorial if you are unsure about how to visualize the structure in mofography

In [ ]:
import plotly.graph_objects as go
fig = go.Figure(data=[isosurface_trace, mgr.plot_cell(mat_updated), *mgr.plot_framework_as_mesh(mat_updated)]).update(layout=mgr.dark_layout)


Note: Don't forget teh `*` infront of the framework trace to unpack the list of traces into the figure.

In [ ]:
fig.show(renderer='notebook_connected')

### Tips and tricks #5

Use the stylize flag to stylize the framework and isosurface plot using a default set of lighting options.

In [ ]:
isosurface_trace = mgr.plot_surface_dgrid(cp.asnumpy(dgrid_gpu), ase_atoms = mat_atoms , isovalue=2.5, color='turquoise', opacity=0.6, stylize=True)
frame_trace = mgr.plot_framework_as_mesh(mat_updated, stylize=True)
cell_trace = mgr.plot_cell(mat_updated)
fig = go.Figure(data=[isosurface_trace, cell_trace, *frame_trace]).update(layout=mgr.dark_layout)

In [ ]:
fig.show(renderer='notebook_connected')

### Tips and tricks #6

Use the `stepsize` parameter to plot a coarser surface. If you have a really fine grid, or a really large unit cell, or you simply want to plot the surface quickly, you might want to plot a coarser isosurface. For example, set `stepsize=2` to plot every other vertex in the isosurface.

In [ ]:
isosurface_trace = mgr.plot_surface_dgrid(cp.asnumpy(dgrid_gpu), stepsize=2, ase_atoms = mat_atoms , isovalue=2.5, color='salmon', opacity=0.6, stylize=True)
fig = go.Figure(data=[isosurface_trace, cell_trace, *frame_trace]).update(layout=mgr.white_layout)
fig.show(renderer='notebook_connected')